# So sánh Embedding Models cho PACS++ (Luận văn)

**Mục đích**: Benchmark 5 embedding models trên 75 báo cáo y tế tiếng Việt thật.

**Models**:
1. BGE-M3 (BAAI) — 1024d
2. multilingual-e5-large (Microsoft) — 1024d
3. GTE-multilingual-base (Alibaba) — 768d
4. paraphrase-multilingual-MiniLM-L12 (Microsoft) — 384d
5. NV-Embed-v2 (NVIDIA) — 4096d

**Metrics**: Precision@5, MRR, nDCG@10, Thời gian encode

In [ ]:
!pip install -q FlagEmbedding sentence-transformers

In [ ]:
import time
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

print('Libraries loaded OK')

## 1. Load Data (75 báo cáo y tế tiếng Việt)

In [ ]:
# Load reports_data.json
# Kaggle: upload file JSON vao Dataset, path se la /kaggle/input/...
DATA_PATHS = [
    '/kaggle/input/pacs-reports/reports_data.json',
    './reports_data.json',
    '../input/pacs-reports/reports_data.json',
]

reports = None
for p in DATA_PATHS:
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            reports = json.load(f)
        print(f'Loaded {len(reports)} reports from {p}')
        break

if reports is None:
    raise FileNotFoundError('Khong tim thay reports_data.json! Upload len Kaggle Dataset truoc.')

# Xem sample
print(f"\nSample report #{reports[0]['id']}:")
print(f"  Modality: {reports[0].get('modality')}")
print(f"  Findings: {reports[0]['findings'][:100]}...")
print(f"  Conclusion: {reports[0]['conclusion'][:100]}...")

## 2. Tạo Ground Truth Queries

In [ ]:
def find_ids(keyword, reports):
    """Tim report IDs chua keyword trong findings/conclusion"""
    ids = []
    for r in reports:
        text = f"{r.get('findings','')} {r.get('conclusion','')}"
        if keyword.lower() in text.lower():
            ids.append(r['id'])
    return ids

QUERIES = []

# Chest queries
ids = find_ids('phổi kẽ', reports)
if ids: QUERIES.append({'query': 'tổn thương phổi dạng mờ kính rải rác', 'expected_ids': ids[:10], 'cat': 'chest'})

ids = find_ids('tim to', reports)
if ids: QUERIES.append({'query': 'bóng tim to trên X-quang ngực', 'expected_ids': ids[:8], 'cat': 'chest'})

ids = find_ids('vôi hóa', reports)
if ids: QUERIES.append({'query': 'vôi hóa thành quai động mạch chủ', 'expected_ids': ids[:8], 'cat': 'chest'})

ids = list(set(find_ids('dịch màng phổi', reports) + find_ids('tràn dịch', reports)))
if ids: QUERIES.append({'query': 'tràn dịch màng phổi', 'expected_ids': ids, 'cat': 'chest'})

# Breast queries
ids = find_ids('BI-RADS 5', reports)
if ids: QUERIES.append({'query': 'khối u vú nghi ngờ ác tính', 'expected_ids': ids, 'cat': 'breast'})

ids = find_ids('BI-RADS 3', reports)
if ids: QUERIES.append({'query': 'nốt đặc vú có thể lành tính cần theo dõi', 'expected_ids': ids, 'cat': 'breast'})

ids = find_ids('BI-RADS 4', reports)
if ids: QUERIES.append({'query': 'vi vôi hóa nghi ngờ ác tính trên mammography', 'expected_ids': ids, 'cat': 'breast'})

# Abdomen queries
ids = find_ids('HCC', reports)
if ids: QUERIES.append({'query': 'nốt gan nghi ung thư biểu mô tế bào gan', 'expected_ids': ids, 'cat': 'abdomen'})

ids = find_ids('sỏi', reports)
if ids: QUERIES.append({'query': 'sỏi thận gây giãn đài bể thận ứ nước', 'expected_ids': ids, 'cat': 'abdomen'})

# Spine queries
ids = find_ids('thoát vị', reports)
if ids: QUERIES.append({'query': 'thoát vị đĩa đệm cột sống thắt lưng chèn ép rễ', 'expected_ids': ids, 'cat': 'spine'})

ids = find_ids('thoái hóa', reports)
if ids: QUERIES.append({'query': 'thoái hóa cột sống đa tầng gai xương', 'expected_ids': ids, 'cat': 'spine'})

# Kidney
ids = find_ids('nang đơn thuần', reports)
if ids: QUERIES.append({'query': 'nang đơn thuần thận Bosniak', 'expected_ids': ids, 'cat': 'abdomen'})

print(f'Tong: {len(QUERIES)} queries')
for q in QUERIES:
    print(f"  [{q['cat']:>8}] \"{q['query'][:45]}\" → {len(q['expected_ids'])} relevant docs")

## 3. Helper Functions

In [ ]:
def make_text(report):
    return f"{report.get('findings','')} {report.get('conclusion','')}"

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def compute_metrics(rankings, expected_ids):
    results = {}
    # Precision@5
    top5 = rankings[:5]
    results['P@5'] = len([r for r in top5 if r in expected_ids]) / min(5, len(expected_ids))
    # MRR
    results['MRR'] = 0.0
    for i, rid in enumerate(rankings):
        if rid in expected_ids:
            results['MRR'] = 1.0 / (i + 1)
            break
    # nDCG@10
    dcg = sum(1.0/np.log2(i+2) for i, rid in enumerate(rankings[:10]) if rid in expected_ids)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(10, len(expected_ids))))
    results['nDCG@10'] = dcg / idcg if idcg > 0 else 0.0
    return results

def benchmark_model(name, encode_fn, reports, queries):
    print(f'\n{"="*60}')
    print(f'  Model: {name}')
    print(f'{"="*60}')
    
    # Encode reports
    texts = [make_text(r) for r in reports]
    t0 = time.time()
    report_vecs = encode_fn(texts)
    encode_time = (time.time() - t0) / len(texts)
    dim = len(report_vecs[0])
    print(f'  Dim: {dim}, Encode: {encode_time:.3f}s/report')
    
    # Test queries
    all_metrics = []
    for q in queries:
        t0 = time.time()
        qvec = encode_fn([q['query']])[0]
        qt = time.time() - t0
        
        scores = [(r['id'], cosine_sim(qvec, rv)) for r, rv in zip(reports, report_vecs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        rankings = [s[0] for s in scores]
        
        m = compute_metrics(rankings, q['expected_ids'])
        m['query_time'] = qt
        all_metrics.append(m)
        print(f"  Q: \"{q['query'][:40]}\" MRR={m['MRR']:.2f} P@5={m['P@5']:.2f}")
    
    avg = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
    print(f'  --- AVG: P@5={avg["P@5"]:.3f} MRR={avg["MRR"]:.3f} nDCG@10={avg["nDCG@10"]:.3f}')
    
    return {
        'model': name, 'dim': dim,
        'P@5': round(avg['P@5'], 3),
        'MRR': round(avg['MRR'], 3),
        'nDCG@10': round(avg['nDCG@10'], 3),
        'query_ms': round(avg['query_time']*1000, 1),
        'encode_ms': round(encode_time*1000, 1),
    }

print('Functions ready')

## 4. Benchmark: BGE-M3 (1024d)

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model_bge = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def encode_bge(texts):
    out = model_bge.encode(texts, batch_size=8, max_length=512)
    return [v.tolist() for v in out['dense_vecs']]

r_bge = benchmark_model('BGE-M3 (1024d)', encode_bge, reports, QUERIES)

## 5. Benchmark: multilingual-e5-large (1024d)

In [ ]:
from sentence_transformers import SentenceTransformer
model_e5 = SentenceTransformer('intfloat/multilingual-e5-large')

def encode_e5(texts):
    return model_e5.encode([f'query: {t}' for t in texts]).tolist()

r_e5 = benchmark_model('multilingual-e5-large (1024d)', encode_e5, reports, QUERIES)

## 6. Benchmark: GTE-multilingual-base (768d)

In [ ]:
model_gte = SentenceTransformer('Alibaba-NLP/gte-multilingual-base')

def encode_gte(texts):
    return model_gte.encode(texts).tolist()

r_gte = benchmark_model('GTE-multilingual-base (768d)', encode_gte, reports, QUERIES)

## 7. Benchmark: MiniLM-L12 (384d)

In [ ]:
model_mini = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def encode_mini(texts):
    return model_mini.encode(texts).tolist()

r_mini = benchmark_model('MiniLM-L12 (384d)', encode_mini, reports, QUERIES)

## 8. Benchmark: NV-Embed-v2 (4096d)
⚠️ Cần GPU 16GB+. Nếu không đủ VRAM, skip cell này.

In [ ]:
try:
    model_nv = SentenceTransformer('nvidia/NV-Embed-v2', trust_remote_code=True)
    def encode_nv(texts):
        return model_nv.encode(texts).tolist()
    r_nv = benchmark_model('NV-Embed-v2 (4096d)', encode_nv, reports, QUERIES)
except Exception as e:
    print(f'[SKIP] NV-Embed-v2: {e}')
    r_nv = None

## 9. Bảng kết quả tổng hợp

In [ ]:
all_results = [r_bge, r_e5, r_gte, r_mini]
if r_nv: all_results.append(r_nv)

df = pd.DataFrame(all_results)
df = df[['model', 'dim', 'P@5', 'MRR', 'nDCG@10', 'query_ms', 'encode_ms']]
df.columns = ['Model', 'Dim', 'P@5', 'MRR', 'nDCG@10', 'Query (ms)', 'Encode (ms)']

print(f'\nData: {len(reports)} bao cao y te tieng Viet, {len(QUERIES)} truy van test')
print('='*85)
df

## 10. Biểu đồ so sánh (cho luận văn)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = df['Model'].values
colors = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444', '#8b5cf6']
x = range(len(models))

# Chart 1: P@5
axes[0].bar(x, df['P@5'], color=colors[:len(models)])
axes[0].set_title('Precision@5', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.split('(')[0].strip() for m in models], rotation=25, ha='right')
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(df['P@5']): axes[0].text(i, v+0.02, f'{v:.3f}', ha='center')

# Chart 2: MRR
axes[1].bar(x, df['MRR'], color=colors[:len(models)])
axes[1].set_title('Mean Reciprocal Rank (MRR)', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([m.split('(')[0].strip() for m in models], rotation=25, ha='right')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(df['MRR']): axes[1].text(i, v+0.02, f'{v:.3f}', ha='center')

# Chart 3: nDCG@10
axes[2].bar(x, df['nDCG@10'], color=colors[:len(models)])
axes[2].set_title('nDCG@10', fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels([m.split('(')[0].strip() for m in models], rotation=25, ha='right')
axes[2].set_ylim(0, 1.05)
for i, v in enumerate(df['nDCG@10']): axes[2].text(i, v+0.02, f'{v:.3f}', ha='center')

plt.suptitle('So sánh Embedding Models - Báo cáo y tế tiếng Việt (PACS++)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('embedding_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n[SAVED] embedding_benchmark.png')

In [ ]:
# Bieu do thoi gian
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(models))
width = 0.35
ax.bar([i - width/2 for i in x], df['Query (ms)'], width, label='Query (ms)', color='#3b82f6')
ax.bar([i + width/2 for i in x], df['Encode (ms)'], width, label='Encode (ms)', color='#f59e0b')
ax.set_title('Thời gian xử lý (ms)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.split('(')[0].strip() for m in models], rotation=25, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('embedding_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print('[SAVED] embedding_latency.png')

## 11. Lưu kết quả

In [ ]:
# Save JSON
with open('benchmark_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# Save CSV
df.to_csv('benchmark_results.csv', index=False)

print('[SAVED] benchmark_results.json')
print('[SAVED] benchmark_results.csv')
print('\nDone! Dung ket qua nay cho luan van.')